# Notebook 02c — SmolLM2 three-shot baseline pilot

This notebook evaluates the **untouched** `HuggingFaceTB/SmolLM2-360M` on **300 fixed questions**: 100 GSM8K, 100 SVAMP, and 100 arithmetic. Each prompt contains the same three worked training examples. Training is a later notebook.

The reviewed Notebook 02b stopping and scoring rules are pinned by checksum. This is a development comparison on previously inspected test sets; it is not a new untouched final exam. Accuracy is reported separately for each dataset.

**Kaggle setup:** enable Internet, select a GPU (one T4 is sufficient), and attach the **original Notebook 01 saved output** using Add Input. Run All and Save Version with outputs. You do not need to attach or rerun Notebook 02 or 02b.


## 1. Fetch the shared code from GitHub

Every run records the exact commit. Changes to shared Python code arrive when this cell runs; importing a notebook creates a copy of its launcher cells.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)

Repository: https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git
Code commit: 80439bf16091bf607eddfc152d02d1d47b01e780
Project checkout: /tmp/math-project-z7kjkfj8


From https://github.com/sahan24854/Fine-Tuning-LLM-for-math
 * branch            main       -> FETCH_HEAD
 * [new branch]      main       -> origin/main


## 2. Verify the original bundle and freeze the question selection

GSM8K and SVAMP questions are ranked by SHA-256 of seed, split, and ID. The first 100 in each ranking are selected without consulting model correctness. Arithmetic uses the same ranking within ten strata: five operations × two number ranges, with ten questions from each.

`selection.json` is saved **before loading model weights**. It records question IDs and hashes of the prompts and references, the original bundle fingerprint, frozen demonstrations, base model revision, and reviewed protocol. Keep this file for the later before/after comparison.


In [2]:
import hashlib
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings before continuing.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
INPUT_ROOT = Path(os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input"))
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), str(INPUT_ROOT)], text=True).strip())
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/verify_bundle.py"), str(BUNDLE)], check=True)
PROTOCOL_PATH = PROJECT / "configs/pilot_v2.json"
PILOT = json.loads(PROTOCOL_PATH.read_text())
SOURCE_MANIFEST = json.loads((BUNDLE / "manifest.json").read_text())
BUNDLE_HASH = hashlib.sha256((BUNDLE / "manifest.json").read_bytes()).hexdigest()
PILOT_HASH = hashlib.sha256(PROTOCOL_PATH.read_bytes()).hexdigest()
OUTPUT = Path("/kaggle/working") / f"pilot-v2-{BUNDLE_HASH[:12]}-{PILOT_HASH[:8]}-{CODE_COMMIT[:8]}"
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/run_pilot.py"),
                    "--bundle", str(BUNDLE), "--protocol", str(PROTOCOL_PATH),
                    "--output", str(OUTPUT), "--plan-only"], check=True, cwd=PROJECT)
SELECTION = json.loads((OUTPUT / "selection.json").read_text())
print("Verified bundle:", BUNDLE)
print("Pinned model:", SOURCE_MANIFEST["model"])
print("Prediction count:", SELECTION["prediction_count"])
print("Mode: three-shot (few_shot); maximum new tokens: 512")
print("Comparison contract:", SELECTION["comparison_contract_sha256"])
print("Output:", OUTPUT)


Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04
Verified 17 files; code commit 6056c2f6cd6e096faf544ff026ba230f32639f9a
{
  "predictions": 300,
  "scope": "development_comparison_previously_inspected_test_sets",
  "comparison_contract_sha256": "b34b624c4aa58b70efe3c71d011075b8cc8f6f815631556e1553abe8f4050051",
  "selection": "/kaggle/working/pilot-v2-b9d3419aa714-d7b6ded7-80439bf1/selection.json"
}
Verified bundle: /kaggle/input/notebooks/sahan24854/01-data-and-protocol/data-protocol-20261019T082909Z-1cb02106
Pinned model: {'id': 'HuggingFaceTB/SmolLM2-360M', 'revision': 'f8027fd0eaeea54caa13c31d31b9fdc459c38b49', 'revision_policy': 'Resolve once in notebook 01; every subsequent notebook must use config.resolved.json.'}
Prediction count: 300
Mode: three-shot (few_shot); maximum new tokens: 512
Comparison contract: b34b624c4aa58b70efe3c71d011075b8cc8f6f815631556e1553abe8f4050051
Output: /kaggle/working/pilot-v2-b9d3419aa714-d7b6ded7-80439bf1


## 3. Prepare the inference environment

Use the previously tested isolated PyTorch/CUDA setup. The notebook kernel packages are not replaced. One GPU is used; a second T4 is not used automatically.

On some Kaggle images, the nonfatal `sitecustomize`/`wrapt` warning and cosmetic `torch_dtype` deprecation warning may appear. Installation, CUDA execution, and the contract tests must still succeed before evaluation.


In [3]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-pilot-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
                    "--environment", str(ENVIRONMENT), "--profile", "baseline"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
probe = """
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable: check Kaggle GPU settings.')
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
print('CUDA execution passed')
"""
_ = subprocess.run([str(PYTHON), "-I", "-c", probe], check=True)

Creating isolated environment with Python 3.13.15


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Processing /tmp/math-pip-bootstrap-g7e930ts/pip-25.3-py3-none-any.whl


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 8.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 5.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 9.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 12.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 15.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 8.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 9.7 MB/s  0:00:00



...

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 131.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 20.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 156.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 66.4 MB/s  0:00:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 98.2 MB/s  0:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 100.9 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 151.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 138.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 141.9 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 139.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/905.2 MB 19.9 MB/s  0:00:16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 MB 83.

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


No broken requirements found.
Environment ready: /tmp/math-pilot-env-to11beua/bin/python


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


PyTorch: 2.7.1+cu118 CUDA: 11.8 GPU: Tesla T4
CUDA execution passed


## 4. Check selection, stopping, scoring, and resume behavior

CPU contract tests check complete numbers such as 550, independent stopping per row, exclusion of later invented questions, normalization without changing mathematical correctness, balanced deterministic selection, and safe recovery after interruption. Synthetic local models in tests do not measure SmolLM2 maths performance.


In [4]:
for test_file in ("test_protocol_v2.py", "test_pilot.py"):
    _ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", test_file, "-b", "-v"],
                        cwd=PROJECT, check=True)


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
test_completed_demonstrations_in_prompt_never_trigger_a_stop (test_protocol_v2.PerSequenceStoppingTests.test_completed_demonstrations_in_prompt_never_trigger_a_stop) ... ok
test_diagnostic_cli_loads_saved_model_and_emits_review_only_results (test_protocol_v2.PerSequenceStoppingTests.test_diagnostic_cli_loads_saved_model_and_emits_review_only_results) ... ok
test_huggingface_generate_stops_rows_independently_and_keeps_full_number (test_protocol_v2.PerSequenceStoppingTests.test_huggingface_generate_stops_rows_independently_and_keeps_full_number) ... ok
test_real_model_validation_diagnostic_executes (test_protocol_v2.PerSequenceStoppingTests.test_real_model_validation_diagnostic_executes) ... ok
test_completed_answer_precedes_later_problem_and_labels (test_protocol_v2.ProtocolV2Tests.test_completed_answer_precedes_later_problem_and_labels) ... ok
test_declared_normalization_only (test_pro

## 5. Evaluate the untouched base model — 300 predictions

The model stops at the first completed final-answer line, before a new question, at EOS, or at the 512-token cap. Output formatting is normalized using the reviewed v2 rules. Invalid answers count as incorrect.

Predictions are flushed after each batch of four. If this cell is interrupted in the same session, rerun it to resume. Do not change the code, environment, bundle, or protocol midway through a run. A fresh Kaggle session may have a different runtime identity; keep the saved output even if a new run is required.

The report includes accuracy with descriptive 95% Wilson intervals, answer parseability, cap-hit counts, generated tokens, and the raw answers. Low accuracy is a result of interest, not by itself a reason to repeat or change the test. Runtime depends on repetition and question lengths; this notebook does not extrapolate a completion guarantee from ten validation questions.


In [5]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/run_pilot.py"),
                    "--bundle", str(BUNDLE), "--protocol", str(PROTOCOL_PATH),
                    "--output", str(OUTPUT)], check=True, cwd=PROJECT)
SUMMARY = json.loads((OUTPUT / "summary.json").read_text())
if not SUMMARY["execution_complete"]:
    raise RuntimeError("Pilot is incomplete; rerun this cell with the same settings to resume.")
print("Completed predictions:", SUMMARY["predictions"])
print("Model role:", SUMMARY["model_role"])


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Loading HuggingFaceTB/SmolLM2-360M at f8027fd0eaeea54caa13c31d31b9fdc459c38b49 on Tesla T4


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded in 3.7s
Three-shot pilot: 0/300 predictions saved
Saved 4/300 predictions (gsm8k_test)
Saved 8/300 predictions (gsm8k_test)
Saved 12/300 predictions (gsm8k_test)
Saved 16/300 predictions (gsm8k_test)
Saved 20/300 predictions (gsm8k_test)
Saved 24/300 predictions (gsm8k_test)
Saved 28/300 predictions (gsm8k_test)
Saved 32/300 predictions (gsm8k_test)
Saved 36/300 predictions (gsm8k_test)
Saved 40/300 predictions (gsm8k_test)
Saved 44/300 predictions (gsm8k_test)
Saved 48/300 predictions (gsm8k_test)
Saved 52/300 predictions (gsm8k_test)
Saved 56/300 predictions (gsm8k_test)
Saved 60/300 predictions (gsm8k_test)
Saved 64/300 predictions (gsm8k_test)
Saved 68/300 predictions (gsm8k_test)
Saved 72/300 predictions (gsm8k_test)
Saved 76/300 predictions (gsm8k_test)
Saved 80/300 predictions (gsm8k_test)
Saved 84/300 predictions (gsm8k_test)
Saved 88/300 predictions (gsm8k_test)
Saved 92/300 predictions (gsm8k_test)
Saved 96/300 predictions (gsm8k_test)
Saved 100/300 predictions (

## 6. Inspect the report and save the pilot for comparison

Expand raw predictions to distinguish mathematical errors from missing answers and repetitive output. The headline accuracy uses all questions; no invalid answers are dropped.

Save Version with outputs and retain the ZIP. The later fine-tuned-model evaluation must reuse this selection, source bundle, demonstrations, prompt text, stopping, and scoring. Use training/validation data to choose checkpoints and hyperparameters. Because the original test outcomes have already been inspected, this pilot remains a development comparison; an additional untouched evaluation is needed for a stronger final claim.


In [6]:
from IPython.display import HTML, display
RESULT = json.loads((OUTPUT / "result_manifest.json").read_text())
for name, expected in RESULT["files_sha256"].items():
    if hashlib.sha256((OUTPUT / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Pilot result checksum mismatch: {name}")
display(HTML((OUTPUT / "report.html").read_text()))
ARCHIVE = Path(shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT))
print("Pilot ZIP:", ARCHIVE)
print("Save Version with outputs. Keep selection.json and the original Notebook 01 bundle for the training comparison.")


Dataset,Count,Accuracy (95% Wilson interval),Parse rate,Cap hits
gsm8k_test,100/100,5.0% (2.2%–11.2%),75.0%,16
svamp_test,100/100,20.0% (13.3%–28.9%),84.0%,9
arithmetic_test,100/100,4.0% (1.6%–9.8%),23.0%,8


Pilot ZIP: /kaggle/working/pilot-v2-b9d3419aa714-d7b6ded7-80439bf1.zip
Save Version with outputs. Keep selection.json and the original Notebook 01 bundle for the training comparison.
